# MIS/MVC QAOA – Parameterized LX Initial State

This notebook demonstrates `MIS_MVC_ParameterizedInitialState`, which exposes
the LX-sweep rotation angles as variational parameters.

Three **grouping** schemes are supported:

| Scheme | `n_params` | Notes |
|---|---|---|
| `"uniform"` | 1 | One shared angle θ for all vertices – most scalable |
| `"degree"` | # distinct degrees | One angle per degree bucket – exploits graph symmetry |
| `"per_vertex"` | N | One angle per qubit – maximum expressivity |

The helper `mis_mvc_warm_start_angle` returns principled initial values based
on the mean-field argument `θ_k = arctan(1/√d_k)`, which is independent of
system size and transfers well across graph sizes via parameter concentration.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from qaoa import QAOA, initializers
from qaoa.initialstates import MIS_MVC_ParameterizedInitialState
from qaoa.mixers import VertexSubsetLXMixer
from qaoa.problems import MIS_MVC_Problem
from qaoa.utils.vertex_subset import mis_mvc_warm_start_angle

## Graph

We use the same small graph from https://arxiv.org/abs/2607.27915 as the
existing `ToyExample` notebook.

In [ ]:
G = nx.Graph()
G.add_edges_from([(0, 1), (0, 4), (1, 2), (2, 3), (3, 4)])

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos=pos, with_labels=True, node_color="lightblue", node_size=600)
plt.title("Problem graph")
plt.show()

print("Nodes:", list(G.nodes()))
print("Degrees:", dict(G.degree()))

## Warm-start angles

`mis_mvc_warm_start_angle` computes `θ = arctan(1/√d)` for each degree
bucket, motivated by the mean-field saddle-point for feasibility-preserving
product states.  These values are graph-size independent and carry over to
larger instances.

In [ ]:
theta_uniform = mis_mvc_warm_start_angle(G, grouping="uniform")
theta_degree  = mis_mvc_warm_start_angle(G, grouping="degree")
theta_vertex  = mis_mvc_warm_start_angle(G, grouping="per_vertex")

print(f"uniform   → θ = {theta_uniform:.4f}  ({np.degrees(theta_uniform):.1f}°)")
print(f"degree    → θ per bucket = {[f'{a:.4f}' for a in theta_degree]}")
print(f"per_vertex→ θ per qubit  = {[f'{a:.4f}' for a in theta_vertex]}")

## Circuit inspection

The parameterized initial-state circuit has Qiskit `Parameter` objects
wherever fixed angles appeared before.

In [ ]:
for grouping in ("uniform", "degree", "per_vertex"):
    init = MIS_MVC_ParameterizedInitialState(G, problem_kind="mis", grouping=grouping)
    init.create_circuit()
    print(f"grouping={grouping!r:12s}  n_params={init.get_num_parameters()}  "
          f"params={[p.name for p in init.circuit.parameters]}")

## QAOA optimization – uniform grouping

A single variational initial-state angle θ (uniform grouping) is the most
scalable choice.  We warm-start it at `θ₀ = arctan(1/√mean_degree)` via
`FixedAngles`.

The flat angle array is `[θ, gamma_0, beta_0, …]`, so the warm-start vector
needs a leading element for θ followed by dummy QAOA angles (zeros here;
the optimiser refines them from the grid).

In [ ]:
problem  = MIS_MVC_Problem(G, problem_kind="mis")
mixer    = VertexSubsetLXMixer(G, problem_kind="mis")
init     = MIS_MVC_ParameterizedInitialState(G, problem_kind="mis", grouping="uniform")

# Build a warm-start: [theta_0=arctan(1/sqrt(mean_deg)), gamma=0, beta=0]
warm_start = np.array([theta_uniform, 0.0, 0.0])

qaoa = QAOA(
    problem,
    mixer,
    init,
    shots=2048,
    cvar=0.2,
    initializer=initializers.FixedAngles(warm_start),
)

qaoa.optimize(depth=1)
print("Best objective (MIS size):", qaoa.get_objective(1))
sols, energy = qaoa.optimization_results[1].get_best_solution()
print("Best solution(s):", list(sols))

## Comparing groupings at depth 1

We run all three groupings and compare the best CVaR energy.

In [ ]:
results = {}
warm_starts = {
    "uniform":    np.array([theta_uniform, 0.0, 0.0]),
    "degree":     np.concatenate([theta_degree,  [0.0, 0.0]]),
    "per_vertex": np.concatenate([theta_vertex,  [0.0, 0.0]]),
}

for grouping in ("uniform", "degree", "per_vertex"):
    init = MIS_MVC_ParameterizedInitialState(G, problem_kind="mis", grouping=grouping)
    qaoa = QAOA(
        problem,
        mixer,
        init,
        shots=2048,
        cvar=0.2,
        initializer=initializers.FixedAngles(warm_starts[grouping]),
    )
    qaoa.optimize(depth=1)
    results[grouping] = {
        "energy": qaoa.get_energy(1),
        "objective": qaoa.get_objective(1),
        "n_params": init.get_num_parameters() + 2,  # incl. gamma + beta
    }
    print(f"{grouping:12s}  n_params={results[grouping]['n_params']}  "
          f"CVaR={results[grouping]['energy']:.4f}  "
          f"objective={results[grouping]['objective']:.4f}")

## Scaling: warm-start angle transferability

The mean-field angle `θ = arctan(1/√d)` depends only on degree statistics,
not on N.  Below we verify that the uniform warm-start found on the 5-node
graph also gives a good starting point for a larger Erdős–Rényi graph with
similar average degree.

In [ ]:
# Larger graph with similar average degree (~2.0)
G_large = nx.erdos_renyi_graph(n=12, p=0.2, seed=7)
print(f"Large graph: {G_large.number_of_nodes()} nodes, "
      f"mean degree={2*G_large.number_of_edges()/G_large.number_of_nodes():.2f}")

theta_large = mis_mvc_warm_start_angle(G_large, grouping="uniform")
print(f"Warm-start angle on large graph: {theta_large:.4f}  "
      f"(original: {theta_uniform:.4f})")

problem_large = MIS_MVC_Problem(G_large, problem_kind="mis")
mixer_large   = VertexSubsetLXMixer(G_large, problem_kind="mis")
init_large    = MIS_MVC_ParameterizedInitialState(G_large, problem_kind="mis",
                                                  grouping="uniform")

# Transfer the angle found on the small graph as the warm-start
transferred_warm_start = np.array([theta_uniform, 0.0, 0.0])

qaoa_large = QAOA(
    problem_large,
    mixer_large,
    init_large,
    shots=2048,
    cvar=0.2,
    initializer=initializers.FixedAngles(transferred_warm_start),
)
qaoa_large.optimize(depth=1)
print("Large graph – best objective (MIS size):", qaoa_large.get_objective(1))